# Exploratory data analysis on OpenNeuro data

### Atif M. Mahmud

### Introduction

This data has been downloaded from https://openneuro.org/datasets/ds003838/versions/1.0.6

This excerpt is from the webpage:
> 
> This dataset consists of raw 64-channel EEG, cardiovascular (electrocardiography and photoplethysmography), and pupillometry data from 86 human participants during 4 minutes of eyes-closed resting and during performance of a
> classic working memory task – digit span task with serial recall. The participants either memorized (memory) or just listened to (control condition) sequences of 5, 9, or 13 digits presented auditorily with 2 second stimulus
> onset asynchrony. The dataset can be used for (1) developing algorithms for cognitive load discrimination and detection of cognitive overload; (2) studying neural (event-related potentials and brain oscillations) and
> peripheral physiological (electrocardiography, photoplethysmography, and pupillometry) signals during encoding and maintenance of each sequentially presented memory item in a fine time scale; (3) correlating cognitive load and > individual differences in working memory to neural and peripheral physiology, and studying the relationship between the physiological signals; (4) integration of the physiological findings with the vast knowledge coming from
> behavioral studies of verbal working memory in simple span paradigms.
> 
> EEG, pupillometry, ECG and photoplethysmography, and behavioral data are stored separately in corresponding folders. Each data record can consist of four data folders:  
> - beh - behavioral data: correctness of the recall in the memory trials
> - ecg - electrocardiography (ECG)
> - photoplethysmography (PPG) data
> - eeg - EEG data
> - pupil - pupillometry and eye-tracking data
> 
> Some of the participants had some physiological data missing: sub-017, sub-094 have no pupillometry data sub-017, sub-037, sub-066 have no ECG and PPG data sub-013, sub-014, sub-015, sub-016, sub-017, sub-018, sub-019,
> sub-020, sub-021, sub-022, sub-023, sub-024, sub-025, sub-026, sub-027, sub-028, sub-029, sub-030, sub-031, sub-037, sub-066 have no EEG data

In [ ]:
import mne
import pandas as pd
import csv
import matplotlib.pyplot as plt
import warnings
import os
import numpy as np
from mne_icalabel import label_components
from IPython.display import display
import traceback
from sklearn.model_selection import RandomizedSearchCV, LeaveOneGroupOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=UserWarning, module="pymatreader")

c:\Users\atifm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Our analysis

There are 86 participants. We won't use the following because they have missing data.  
- sub-013 to sub-031
- sub-037
- sub-066
- sub-094

In [5]:
# The EEG data is in the `_eeg.set` files
# %matplotlib qt

raw_sub40_task = mne.io.read_raw_eeglab("data/sub-040/eeg/sub-040_task-memory_eeg.set", preload=True)
print(f"The shape of the date is {raw_sub40_task.get_data().shape}")
print(f"The channel names are: {raw_sub40_task.ch_names}")

# raw_sub40_task.plot()
# plt.show()

The shape of the date is (63, 7176340)
The channel names are: ['Fp1', 'Fz', 'F3', 'F7', 'FT9', 'FC5', 'FC1', 'C3', 'T7', 'TP9', 'CP5', 'CP1', 'Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'TP10', 'CP6', 'CP2', 'Cz', 'C4', 'T8', 'FT10', 'FC6', 'FC2', 'F4', 'F8', 'Fp2', 'AF7', 'AF3', 'AFz', 'F1', 'F5', 'FT7', 'FC3', 'C1', 'C5', 'TP7', 'CP3', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'P6', 'P2', 'CPz', 'CP4', 'TP8', 'C6', 'C2', 'FC4', 'FT8', 'F6', 'AF8', 'AF4', 'F2']


From participant 40, we see that we are able to load the EEG data using `mne` and `pymatreader`. This is a 64-channel EEG, so we have 63-channels of data since one of the channels is a reference and the other values are recorded in reference to that.

## EEG classification

### Load, filter, and re-reference data

In [ ]:
participants = [32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65]
print(f"N =  {len(participants)}")

cache_dir = "data/filtered-referenced-eeg"
os.makedirs(cache_dir, exist_ok=True)

for participant in participants:
    cached_file = f"{cache_dir}/sub-0{participant}-task-eeg_filtered-referenced.fif"

    if os.path.exists(cached_file):
        print("File exists!")
    else:
        # Cached, pre-processed file not found: load raw file
        file = f"data/sub-0{participant}/eeg/sub-0{participant}_task-memory_eeg.set"
        raw = mne.io.read_raw_eeglab(file, preload=True)

        # Frequency filter: High-pass 1Hz, Low-pass 45Hz & re-reference to averaged reference
        # Based on the original authors' implementation: (Kosachenko et al., 2023)
        raw.filter(l_freq=1, h_freq=45)
        raw.set_eeg_reference("average")

        # Save file to cache
        raw.save(cached_file, overwrite=True)

### ICA and epoching

In [ ]:
participant = 32
cache_dir = "data/filtered-referenced-eeg"
cached_file = f"{cache_dir}/sub-0{participant}-task-eeg_filtered-referenced.fif"

if os.path.exists(cached_file):
    print(f"Found file {cached_file}")
    raw = mne.io.read_raw_fif(cached_file, preload=True)
    ica = mne.preprocessing.ICA(n_components=0.99, method="fastica", random_state=42)
    ica.fit(raw)
    ica.plot_components()
    ica_labels = label_components(raw, ica, "iclabel")
    display(ica_labels)
else:
    print(f"File - {cached_file} not found!")


In [ ]:
display(pd.DataFrame(ica_labels))

#### For each participant remove the artifacts using mne-icalabel

In [ ]:
cache_dir = "data/filtered-referenced-eeg"
target_dir = "data/ica-excluded-eeg"
os.makedirs(target_dir, exist_ok=True)

participants = [32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65]
for participant in participants:
    cached_file = f"{cache_dir}/sub-0{participant}-task-eeg_filtered-referenced.fif"
    target_file = f"{target_dir}/sub-0{participant}-task-eeg_ica-cleaned.fif"
    if os.path.exists(target_file):
        print(f"ICA cleaned file exists for {participant} - {target_file}. Skipping...")
        continue
    if os.path.exists(cached_file):
        try:
            print(f"Found file {cached_file}")
            raw = mne.io.read_raw_fif(cached_file, preload=True)
            ica = mne.preprocessing.ICA(n_components=0.99, method="fastica", random_state=42)
            ica.fit(raw)
            ica_labels = label_components(raw, ica, "iclabel")
            excluded_components = []
            for idx, label in enumerate(ica_labels["labels"]):
                if label != "brain":
                    excluded_components.append(idx)
                    print(f"Excluding index {idx} from {participant} because label is {label}")
            print(f"For participant {participant}, we are excluding components {excluded_components}")
            ica.exclude = excluded_components
            eeg_ica_cleaned = ica.apply(raw.copy())
            eeg_ica_cleaned.save(target_file, overwrite=True)
            print(f"@ Saved file {target_file}")
        except Exception as error:
            print(f"Error {error} while trying to apply ICA to participant {participant}")
    else:
        print(f"File - {cached_file} not found!")

#### Epoching the data and creating the X and Y for random forest

In [ ]:
## LABEL = the event string like x500913
## CODE = a numeric code of the event

def parse_label(label):
    if not label.startswith(("5", "6")) or len(label) > 7 or len(label) < 6:
        print(f"Skipping this label: label {label} is invalid.")
        return None
    condition_flag = label[0]
    if condition_flag not in ("5", "6"):
        print(f"Error: label {label}. Condition flag is invalid")
    condition = "Memory" if condition_flag == "6" else "Listen"
    position = int(label[2:4])
    load = int(label[4:6])
    correct = None
    if condition == "Memory" and len(label) > 6:
        correct = "Correct" if label[6] == "1" else "Incorrect"
    return {"label" : label, "condition" : condition, "position" : position, "load" : load, "correct" : correct}


def compute_power(epochs):
    freq = np.concatenate([np.arange(8, 13, 2), np.arange(30, 45, 2)]) # Alpha and Gamma. We are using 45 because we cutoff after that in our filter. Sparser sampling.
    n_cycles = freq / 2
    powers = epochs.compute_tfr(method="morlet", freqs=freq, n_cycles=n_cycles, decim=4, average=False)
    powers.data = powers.data.astype(np.float32)
    # Compute power relative to a time window
    powers.apply_baseline(baseline=(-1.5, -0.2), mode="logratio")
    
    # Crop to get Alpha (8-12) and Gamma (30-44). Average across time, frequencies, channels
    # This will give me a single value for whole scalp
    # FUTURE TODO: average over regions to get ROI specific values
    alpha_mean = powers.copy().crop(fmin=8, fmax=12).data.mean(axis=(1, 2, 3))
    gamma_mean = powers.copy().crop(fmin=30, fmax=44).data.mean(axis=(1, 2, 3))

    return pd.DataFrame({"alpha_power" : alpha_mean, "gamma_power" : gamma_mean})


preprocessed_eeg_dir = "data/ica-excluded-eeg"
def process_participant_data(num):
    print(f"Processing epoch for participant {num}")
    eeg_file = f"{preprocessed_eeg_dir}/sub-0{num}-task-eeg_ica-cleaned.fif"
    if not os.path.exists(eeg_file):
        print(f"File {eeg_file} not found")
    else:
        raw = mne.io.read_raw_fif(eeg_file, preload=True)
        print(f"Read file {eeg_file}")
        events, event_id = mne.events_from_annotations(raw)
        code_to_label = {v : k for k, v in event_id.items()} # Reverse map. Get a dict where event "code" is KEY and "label" is VALUE

    # Loop through events in time series
    metadata, valid_event_indices = [], []
    for index, event in enumerate(events):
        event_description = parse_label(code_to_label[event[2]]) # Get the label from the event code which is in the 3rd position in "event"
        if event_description:
            metadata.append(event_description)
            valid_event_indices.append(index)
    
    metadata_df = pd.DataFrame(metadata) # Create metadata dataframe
    valid_events = events[valid_event_indices] # Get the list of clean event codes
    matched_event_labels = {k: v for k, v in event_id.items() if parse_label(k)} # Get the dictionary of event labels to codes, but only where it's valid label

    # Create epoch object: from 1.5 before to 3.5 as authors did
    epochs = mne.Epochs(raw, events=valid_events, event_id=matched_event_labels, tmin=-1.5, tmax=3.5, metadata=metadata_df)

    print(f"Start computing power for participant {num}")
    powers = compute_power(epochs)
    participant_df = epochs.metadata.reset_index(drop=True).copy()
    participant_df["alpha_power"] = powers["alpha_power"].values
    participant_df["gamma_power"] = powers["gamma_power"].values
    participant_df["participant"] = num
    print(f"\nBelow is dataframe for participant {num}")
    display(participant_df)
    return participant_df

participants_batch_1 = [32, 33, 34, 35, 36, 38, 39, 40, 41, 42]
participants_batch_2 = [43, 44, 45, 46, 47, 48, 49, 50, 51, 52]
participants_batch_3 = [53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65] # We will leave this out for lack of time

all_participants = []

print(f"Starting batch_1. all_participants shape: {len(all_participants)}")
for participant in participants_batch_1:
    print(f"\n\nProcessing epoch and power for participant {participant}")
    # try:
    all_participants.append(process_participant_data(participant))
    # except Exception as error:
    #    print(f"Skipped participant {participant}. Error {error}.")

print(f"Starting batch_2. all_participants shape: {len(all_participants)}")
for participant in participants_batch_2:
    print(f"\nProcessing epoch and power for participant {participant}")
    try:
        all_participants.append(process_participant_data(participant))
    except Exception as error:
        print(f"Skipped participant {participant}. Error {error}.")

all_participants_df = pd.concat(all_participants, ignore_index=True)
display(all_particpants)



In [ ]:
print(f"all_participants shape is {len(all_participants)}")
display(all_participants[0])
print(f"all_participants_df shape os {all_participants_df.shape}")
all_participants_df.to_csv("eeg-power-features.csv", index=False)

#### Test cases

In [ ]:
## Test the parse function : IT WORKS! :)
# labels = ["x500813", "x500913", "x501013", "x501113", "x501213", "x501313", "x6001050", "x6001051", "x6002050", "x6002051", "x6003050", "x6003051", "x6004050", "x6004051", "x6005050", "x6005051", "x6001090", "x6001091", "x6002090"]
# for label in labels:
#    print(parse_label(label))

## Testing `mne.events_from_annotations`
# test_raw = mne.io.read_raw_fif("data/ica-excluded-eeg/sub-051-task-eeg_ica-cleaned.fif", preload=True)
# events, event_id = mne.events_from_annotations(test_raw)
# print(f"Events : {events}")
# print(f"Event ID: {event_id}")

      "x500813": "control 08/13: listen to digit 8 in 13 digit sequence",
      "x500913": "control 09/13: listen to digit 9 in 13 digit sequence",
      "x501013": "control 10/13: listen to digit 10 in 13 digit sequence",
      "x501113": "control 11/13: listen to digit 11 in 13 digit sequence",
      "x501113": "control 12/13: listen to digit 12 in 13 digit sequence",
      "x501313": "control 13/13: listen to digit 13 (last) in 13 digit sequence",
      "x6001050": "memory 01/05 error: memorize digit 1 (first) in 5 digit sequence; forgotten",
      "x6001051": "memory 01/05 correct: memorize digit 1 (first) in 5 digit sequence; correctly recalled",
      "x6002050": "memory 02/05 error: memorize digit 2 in 5 digit sequence; forgotten",
      "x6002051": "memory 02/05 correct: memorize digit 2 in 5 digit sequence; correctly recalled",
      "x6003050": "memory 03/05 error: memorize digit 3 in 5 digit sequence; forgotten",
      "x6003051": "memory 03/05 correct: memorize digit 3 in 5 digit sequence; correctly recalled",
      "x6004050": "memory 04/05 error: memorize digit 4 in 5 digit sequence; forgotten",
      "x6004051": "memory 04/05 correct: memorize digit 4 in 5 digit sequence; correctly recalled",
      "x6005050": "memory 05/05 error: memorize digit 5 (last) in 5 digit sequence; forgotten",
      "x6005051": "memory 05/05 correct: memorize digit 5 (last) in 5 digit sequence; correctly recalled",
      "x6001090": "memory 01/09 error: memorize digit 1 (first) in 9 digit sequence; forgotten",
      "x6001091": "memory 01/09 correct: memorize digit 1 (first) in 9 digit sequence; correctly recalled",
      "x6002090": "memory 02/09 error: memorize digit 2 in 9 digi

### Random Forest 1: Whole brain alpha and gamma power

#### Load data

In [2]:
df_features = pd.read_csv("data/eeg-power-features.csv")
print(f"DF: {df_features.shape}")
display(df_features)

DF: (29187, 8)


,label,condition,position,load,correct,alpha_power,gamma_power,participant
0,500113,Listen,1,13,NaN,-0.057477,-0.192534,32
1,500213,Listen,2,13,NaN,-0.394398,-0.184071,32
2,500313,Listen,3,13,NaN,-0.412928,-0.205873,32
3,500413,Listen,4,13,NaN,-0.067001,-0.229368,32
4,500513,Listen,5,13,NaN,-0.253803,-0.232337,32
...,...,...,...,...,...,...,...,...
29182,500913,Listen,9,13,NaN,-0.113997,-0.176047,52
29183,501013,Listen,10,13,NaN,-0.120848,-0.268978,52
29184,501113,Listen,11,13,NaN,-0.434566,-0.307241,52
29185,501213,Listen,12,13,NaN,-0.159789,-0.207058,52


In [3]:
X = df_features[["alpha_power", "gamma_power"]]
Y = df_features["condition"]
group = df_features["participant"]
display(Y.value_counts())



condition
Memory    19440
Listen     9747
Name: count, dtype: int64

My dataset has 19440 memory and 9747 listen. Ideally i would like more of an even split. But for this scope I am okay with it.

#### Hyperparameter tuning using RandomizedSearchCV

In [4]:
logo = LeaveOneGroupOut()

param_grid = {
    "n_estimators" : [50, 100, 200],
    "max_depth" : [10, 15, 20],
    "min_samples_split" : [2, 5, 10, 15]
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    param_distributions = param_grid,
    n_iter = 25,
    cv = list(logo.split(X, Y, groups=group)),
    scoring = "f1_macro",
    random_state = 42,
    n_jobs = -1
)

random_search.fit(X, Y)
print(f"The best params: {random_search.best_params_}")
print(f"The best score: {random_search.best_score_}")

The best params: {'n_estimators': 50, 'min_samples_split': 15, 'max_depth': 15}
The best score: 0.5111036600878409


With `n=10` at one run, I got: 
- The best params: {'n_estimators': 50, 'min_samples_split': 15, 'max_depth': 15}  
- The best score: 0.5111036600878409  

#### Running my random forest

In [7]:
best_params = random_search.best_params_
accuracies, true, predicted = [], [], []

for train, test in logo.split(X, Y, groups=group):
    classifier = RandomForestClassifier(**best_params, random_state=42, class_weight="balanced")
    classifier.fit(X.iloc[train], Y.iloc[train])
    prediction = classifier.predict(X.iloc[test])
    accuracy = accuracy_score(Y.iloc[test], prediction)
    accuracies.append(accuracy)
    true.extend(Y.iloc[test])
    predicted.extend(prediction)

print(f"Mean accuracy: {np.mean(accuracies)}")
print(f"Classification report: {classification_report(true, predicted)}")

Mean accuracy: 0.559403915700212
Classification report:               precision    recall  f1-score   support

      Listen       0.35      0.37      0.36      9747
      Memory       0.67      0.65      0.66     19440

    accuracy                           0.56     29187
   macro avg       0.51      0.51      0.51     29187
weighted avg       0.57      0.56      0.56     29187



### Random Forest 2: ROI specific alpha and gamma power

We are going to redo our random forest, but this time with specific brain region powers instead of collapsing it to a global average. See below, changes 

#### Compute powers 

In [ ]:
## LABEL = the event string like x500913
## CODE = a numeric code of the event

### SAME AS RANDOM FOREST 1
def parse_label(label):
    if not label.startswith(("5", "6")) or len(label) > 7 or len(label) < 6:
        print(f"Skipping this label: label {label} is invalid.")
        return None
    condition_flag = label[0]
    if condition_flag not in ("5", "6"):
        print(f"Error: label {label}. Condition flag is invalid")
    condition = "Memory" if condition_flag == "6" else "Listen"
    position = int(label[2:4])
    load = int(label[4:6])
    correct = None
    if condition == "Memory" and len(label) > 6:
        correct = "Correct" if label[6] == "1" else "Incorrect"
    return {"label" : label, "condition" : condition, "position" : position, "load" : load, "correct" : correct}

##################################################################################
# CHANNELS ARE: The channel names are: ['Fp1', 'Fz', 'F3', 'F7', 'FT9', 
# 'FC5', 'FC1', 'C3', 'T7', 'TP9', 'CP5', 'CP1', 'Pz', 'P3', 'P7', 
# 'O1', 'Oz', 'O2', 'P4', 'P8', 'TP10', 'CP6', 'CP2', 'Cz', 'C4', 
# 'T8', 'FT10', 'FC6', 'FC2', 'F4', 'F8', 'Fp2', 'AF7', 'AF3', 'AFz', 
# 'F1', 'F5', 'FT7', 'FC3', 'C1', 'C5', 'TP7', 'CP3', 'P1', 'P5', 'PO7', 
# 'PO3', 'POz', 'PO4', 'PO8', 'P6', 'P2', 'CPz', 'CP4', 'TP8', 'C6', 'C2', 
# 'FC4', 'FT8', 'F6', 'AF8', 'AF4', 'F2']

# Groups defined as per: https://neuroanalyzer.org/tutorials/locs.html
roi_groups = {
    "frontal" : ["Fp1", "Fp2", "F3", "F4", "F7", "F8", "Fz"], # Frontal lobe: Executive function, attention, decision-making, and working memory.
    "parietal" : ["P3", "P4", "P7", "P8", "Pz", "CP1", "CP2", "CP5", "CP6"], # Parietal lobe: Spatial orientation, sensory integration, and attention.
    "temporal": ["T7", "T8", "F7", "F8", "FT9", "FT10"],  # Temporal lobe: Auditory processing, language, and memory.
    "midline" : ["Fz", "Cz", "Pz", "Oz"],  # Midline structures: Default mode network, global brain states.
}

# Thus we will have:
# frontal_alpha, frontal_gamma
# parietal_alpha, parietal_gamma
# temporal_alpha, temporal_gamma
# midline_alpha, midline_gamma

def compute_power(epochs):
    freq = np.concatenate([np.arange(8, 13, 2), np.arange(30, 45, 2)]) # Alpha and Gamma. We are using 45 because we cutoff after that in our filter. Sparser sampling.
    n_cycles = freq / 2
    powers = epochs.compute_tfr(method="morlet", freqs=freq, n_cycles=n_cycles, decim=4, average=False)
    powers.data = powers.data.astype(np.float32)
    # Compute power relative to a time window
    powers.apply_baseline(baseline=(-1.5, -0.2), mode="logratio")
    
    alpha = powers.copy().crop(fmin=8, fmax=12)
    gamma = powers.copy().crop(fmin=30, fmax=44)

    # The key difference here: get ROI specific channels
    # I can do list construction one-line but I like this more
    ch_indices = []
    roi_powers = {}
    for roi_name, ch_list in roi_groups.items():
        ch_indices = []
        for ch in ch_list:
            if ch in powers.ch_names:
                ch_indices.append(powers.ch_names.index(ch))
        roi_powers[f"{roi_name}_alpha"] = alpha.data[:, ch_indices, :, :].mean(axis=(1, 2, 3))
        roi_powers[f"{roi_name}_gamma"] = gamma.data[:, ch_indices, :, :].mean(axis=(1, 2, 3))

    return pd.DataFrame(roi_powers)
##################################################################################

### SAME AS RANDOM FOREST 1
preprocessed_eeg_dir = "data/ica-excluded-eeg"
def process_participant_data(num):
    print(f"Processing epoch for participant {num}")
    eeg_file = f"{preprocessed_eeg_dir}/sub-0{num}-task-eeg_ica-cleaned.fif"
    if not os.path.exists(eeg_file):
        print(f"File {eeg_file} not found")
    else:
        raw = mne.io.read_raw_fif(eeg_file, preload=True)
        print(f"Read file {eeg_file}")
        events, event_id = mne.events_from_annotations(raw)
        code_to_label = {v : k for k, v in event_id.items()} # Reverse map. Get a dict where event "code" is KEY and "label" is VALUE

    # Loop through events in time series
    metadata, valid_event_indices = [], []
    for index, event in enumerate(events):
        event_description = parse_label(code_to_label[event[2]]) # Get the label from the event code which is in the 3rd position in "event"
        if event_description:
            metadata.append(event_description)
            valid_event_indices.append(index)
    
    metadata_df = pd.DataFrame(metadata) # Create metadata dataframe
    valid_events = events[valid_event_indices] # Get the list of clean event codes
    matched_event_labels = {k: v for k, v in event_id.items() if parse_label(k)} # Get the dictionary of event labels to codes, but only where it's valid label

    # Create epoch object: from 1.5 before to 3.5 as authors did
    epochs = mne.Epochs(raw, events=valid_events, event_id=matched_event_labels, tmin=-1.5, tmax=3.5, metadata=metadata_df)

    print(f"Start computing power for participant {num}")

    powers = compute_power(epochs)
    participant_df = epochs.metadata.reset_index(drop=True).copy()

    # Thus we will have:
    # frontal_alpha, frontal_gamma
    # parietal_alpha, parietal_gamma
    # temporal_alpha, temporal_gamma
    # midline_alpha, midline_gamma

    participant_df["frontal_alpha"] = powers["frontal_alpha"].values
    participant_df["frontal_gamma"] = powers["frontal_gamma"].values
    
    participant_df["parietal_alpha"] = powers["parietal_alpha"].values
    participant_df["parietal_gamma"] = powers["parietal_gamma"].values
    
    participant_df["temporal_alpha"] = powers["temporal_alpha"].values
    participant_df["temporal_gamma"] = powers["temporal_gamma"].values
    
    participant_df["midline_alpha"] = powers["midline_alpha"].values
    participant_df["midline_gamma"] = powers["midline_gamma"].values

    participant_df["participant"] = num

    print(f"\nBelow is dataframe for participant {num}")
    display(participant_df)
    return participant_df

participants_batch_1 = [32, 33, 34, 35, 36, 38, 39, 40, 41, 42]
participants_batch_2 = [43, 44, 45, 46, 47, 48, 49, 50, 51, 52]
participants_batch_3 = [53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65] # We will leave this out for lack of time

all_participants_2 = []

print(f"Starting batch_1. all_participants shape: {len(all_participants_2)}")
for participant in participants_batch_1:
    print(f"\n\nProcessing epoch and power for participant {participant}")
    # try:
    all_participants_2.append(process_participant_data(participant))
    # except Exception as error:
    #    print(f"Skipped participant {participant}. Error {error}.")

print(f"Starting batch_2. all_participants shape: {len(all_participants_2)}")
for participant in participants_batch_2:
    print(f"\nProcessing epoch and power for participant {participant}")
    try:
        all_participants_2.append(process_participant_data(participant))
    except Exception as error:
        print(f"Skipped participant {participant}. Error {error}.")

all_participants_df_2 = pd.concat(all_participants_2, ignore_index=True)
display(all_participants_df_2)
all_participants_df_2.to_csv("data/eeg-power-features-per-roi.csv", index=False)

#### Load Data

In [4]:
df_features_2 = pd.read_csv("data/eeg-power-features-per-roi.csv")
print(f"DF: {df_features_2.shape}")
display(df_features_2)

DF: (29187, 14)


,label,condition,position,load,correct,frontal_alpha,frontal_gamma,parietal_alpha,parietal_gamma,temporal_alpha,temporal_gamma,midline_alpha,midline_gamma,participant
0,500113,Listen,1,13,NaN,-0.025957,-0.183265,-0.009031,-0.205845,-0.157359,-0.194499,-0.096440,-0.160752,32
1,500213,Listen,2,13,NaN,-0.429560,-0.158683,-0.399298,-0.258114,-0.363661,-0.172133,-0.402520,-0.166223,32
2,500313,Listen,3,13,NaN,-0.422900,-0.244903,-0.425542,-0.174593,-0.307508,-0.225774,-0.410360,-0.229743,32
3,500413,Listen,4,13,NaN,-0.069365,-0.217401,-0.060179,-0.203852,0.063792,-0.215970,-0.118950,-0.210434,32
4,500513,Listen,5,13,NaN,-0.308645,-0.271917,-0.230197,-0.221062,-0.364749,-0.268459,-0.277217,-0.253627,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29182,500913,Listen,9,13,NaN,-0.010581,-0.196923,-0.168637,-0.183893,-0.322422,-0.167537,-0.125526,-0.169027,52
29183,501013,Listen,10,13,NaN,-0.073557,-0.247344,-0.187208,-0.239437,-0.049886,-0.277497,-0.070308,-0.279973,52
29184,501113,Listen,11,13,NaN,-0.529485,-0.233043,-0.374201,-0.290715,-0.287149,-0.341547,-0.557564,-0.325334,52
29185,501213,Listen,12,13,NaN,-0.076425,-0.231195,-0.211442,-0.188937,-0.180754,-0.186142,-0.172840,-0.222491,52


In [5]:
X = df_features_2[["frontal_alpha", "frontal_gamma",
    "parietal_alpha", "parietal_gamma",
    "temporal_alpha", "temporal_gamma",
    "midline_alpha", "midline_gamma"
]]

Y = df_features_2["condition"]
group = df_features_2["participant"]

display(X)
display(Y)
display(group)

,frontal_alpha,frontal_gamma,parietal_alpha,parietal_gamma,temporal_alpha,temporal_gamma,midline_alpha,midline_gamma
0,-0.025957,-0.183265,-0.009031,-0.205845,-0.157359,-0.194499,-0.096440,-0.160752
1,-0.429560,-0.158683,-0.399298,-0.258114,-0.363661,-0.172133,-0.402520,-0.166223
2,-0.422900,-0.244903,-0.425542,-0.174593,-0.307508,-0.225774,-0.410360,-0.229743
3,-0.069365,-0.217401,-0.060179,-0.203852,0.063792,-0.215970,-0.118950,-0.210434
4,-0.308645,-0.271917,-0.230197,-0.221062,-0.364749,-0.268459,-0.277217,-0.253627
...,...,...,...,...,...,...,...,...
29182,-0.010581,-0.196923,-0.168637,-0.183893,-0.322422,-0.167537,-0.125526,-0.169027
29183,-0.073557,-0.247344,-0.187208,-0.239437,-0.049886,-0.277497,-0.070308,-0.279973
29184,-0.529485,-0.233043,-0.374201,-0.290715,-0.287149,-0.341547,-0.557564,-0.325334
29185,-0.076425,-0.231195,-0.211442,-0.188937,-0.180754,-0.186142,-0.172840,-0.222491


0        Listen
1        Listen
2        Listen
3        Listen
4        Listen
          ...  
29182    Listen
29183    Listen
29184    Listen
29185    Listen
29186    Listen
Name: condition, Length: 29187, dtype: str

0        32
1        32
2        32
3        32
4        32
         ..
29182    52
29183    52
29184    52
29185    52
29186    52
Name: participant, Length: 29187, dtype: int64

#### Hyperparamter tuning

In [10]:
logo = LeaveOneGroupOut()

param_grid = {
    "n_estimators" : [50, 100, 200, 300, 400],
    "max_depth" : [10, 15, 20, 25, 30],
    "min_samples_split" : [2, 5, 10, 15, 20, 25]
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    param_distributions = param_grid,
    n_iter = 25,
    cv = list(logo.split(X, Y, groups=group)),
    scoring = "f1_macro",
    random_state = 42,
    n_jobs = -1,
)

random_search.fit(X, Y)
best_params = random_search.best_params_
print(f"The best params are: {best_params}")
print(f"The best sore is: {random_search.best_score_}")

The best params are: {'n_estimators': 400, 'min_samples_split': 15, 'max_depth': 10}
The best sore is: 0.5152142453829789


In [ ]:
corr_matrix = df_features_2[["frontal_alpha", "frontal_gamma",
    "parietal_alpha", "parietal_gamma",
    "temporal_alpha", "temporal_gamma",
    "midline_alpha", "midline_gamma"
]].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, cmap="coolwarm", fmt=".2f")
plt.show()


,frontal_alpha,frontal_gamma,parietal_alpha,parietal_gamma,temporal_alpha,temporal_gamma,midline_alpha,midline_gamma
frontal_alpha,1.000000,0.056528,0.858266,0.089893,0.842395,0.039117,0.908505,0.085993
frontal_gamma,0.056528,1.000000,0.048565,0.852683,0.071417,0.862415,0.034854,0.854921
parietal_alpha,0.858266,0.048565,1.000000,0.096227,0.854137,0.032209,0.858544,0.083996
parietal_gamma,0.089893,0.852683,0.096227,1.000000,0.105888,0.841100,0.071876,0.820046
temporal_alpha,0.842395,0.071417,0.854137,0.105888,1.000000,0.061324,0.793611,0.094767
temporal_gamma,0.039117,0.862415,0.032209,0.841100,0.061324,1.000000,0.019926,0.722941
midline_alpha,0.908505,0.034854,0.858544,0.071876,0.793611,0.019926,1.000000,0.072804
midline_gamma,0.085993,0.854921,0.083996,0.820046,0.094767,0.722941,0.072804,1.000000
